# ⚡ 04. Hệ Thống Quét Tín Hiệu Thực Chiến EOD (Live Signal Scanner)
### *Tự Động Quét Tín Hiệu Cuối Phiên (14:15 - 14:25) & Xuất Daily Order Sheet*

Notebook này hướng dẫn quy trình vận hành thực chiến hàng ngày:
1. **Nạp trạng thái danh mục hiện tại (`portfolio_state.json`).**
2. **Kiểm tra bộ lọc vĩ mô & vi cấu trúc thị trường (VN30 ADX14, Phái sinh Basis).**
3. **Quét tín hiệu xung lực Volume Spread Analysis (RVOL $\ge 1.6$, Breakout đỉnh 3 phiên).**
4. **Tính toán khối lượng đặt lệnh với khống chế trần 5% ADV20.**
5. **Xuất phiếu lệnh giao dịch (`daily_order_sheet.csv`).**


In [ ]:
import os
import sys
from pathlib import Path
import json
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / "data"

sys.path.append(str(PROJECT_ROOT))
import live_order_engine as live_engine

print("Đã nạp module live_order_engine thành công!")


## 1. Kiểm Tra Trạng Thái Danh Mục Hiện Tại
Xem xét lượng tiền mặt khả dụng, các vị thế cổ phiếu đang nắm giữ, giá vốn và tỷ trọng.


In [ ]:
state_path = DATA_DIR / "5_live_execution" / "portfolio_state.json"
with open(state_path, 'r', encoding='utf-8') as f:
    current_state = json.load(f)

print(f"Tổng tài sản (NAV):      {current_state['cash'] + sum(p['shares'] * p['entry_price'] for p in current_state.get('positions', {}).values()):,.0f} VNĐ")
print(f"Tiền mặt khả dụng:        {current_state['cash']:,.0f} VNĐ")
print(f"Số vị thế đang nắm giữ:  {len(current_state.get('positions', {}))}")

if current_state.get('positions'):
    positions_df = pd.DataFrame.from_dict(current_state['positions'], orient='index')
    display(positions_df)
else:
    print("Hiện tại danh mục đang giữ 100% tiền mặt, sẵn sàng giải ngân khi có tín hiệu mới.")


## 2. Thực Thi Quét Toàn Bộ Rổ VN30 Cuối Phiên
Hàm `run_daily_scanner()` sẽ tự động:
- Kiểm tra xu hướng thị trường (ADX $\ge 18$).
- Kiểm tra chênh lệch Basis phái sinh.
- Quét tín hiệu khối lượng và giá của 30 cổ phiếu.
- Tính toán khối lượng đặt lệnh theo tỷ trọng HRP và khống chế 5% ADV20.


In [ ]:
# Chạy quét tín hiệu thực chiến
signals_df = live_engine.run_daily_scanner(save_sheet=True)

if not signals_df.empty:
    print(f"TÌM THẤY {len(signals_df)} TÍN HIỆU ĐẠT CHUẨN GIẢI NGÂN HÔM NAY:")
    display(signals_df[['ticker', 'action', 'signal_type', 'target_shares', 'est_price', 'est_value_vnd', 'adv20_limit']])
else:
    print("Hôm nay thị trường không thỏa điều kiện hoặc không có cổ phiếu nào đạt chuẩn breakout.")


## 3. Xem Phiếu Lệnh Giao Dịch Đã Xuất (Daily Order Sheet)
File này có thể nạp thẳng vào hệ thống giao dịch tự động hoặc chuyển cho môi giới/trader đặt lệnh vào phiên ATC.


In [ ]:
order_sheet_path = DATA_DIR / "5_live_execution" / "daily_order_sheet.csv"
if order_sheet_path.exists():
    order_sheet = pd.read_csv(order_sheet_path)
    print("PHIẾU LỆNH GIAO DỊCH (DAILY ORDER SHEET):")
    display(order_sheet)
else:
    print("Chưa có phiếu lệnh mới.")
